# Qwen2.5-1.5B QLoRA — Vietnamese Dialogue Rewriter (Kaggle)

Notebook chạy end-to-end trên Kaggle GPU **T4 16GB**.

**Settings (panel bên phải):**
- Accelerator: `GPU T4 x1` (hoặc x2)
- Internet: **On** (bắt buộc để `pip install` và tải Qwen từ HuggingFace)

Trước khi Run All, sửa biến `REPO_URL` ở cell tiếp theo cho đúng repo của bạn.

## 1. Lấy code

Chọn một trong hai cách bằng cách bật `USE_KAGGLE_DATASET`.

- `False` → clone từ GitHub (`REPO_URL`).
- `True` → copy từ Kaggle Dataset đã upload (`KAGGLE_DATASET_DIR`).

In [ ]:
import os, shutil, subprocess

USE_KAGGLE_DATASET = False
REPO_URL = "https://github.com/thao12345310/qwen-lora-finetuning.git"  # sửa lại
KAGGLE_DATASET_DIR = "/kaggle/input/qwen-lora-finetuning"           # sửa lại nếu khác
PROJECT_DIR = "/kaggle/working/qwen-lora-finetuning"

if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)

if USE_KAGGLE_DATASET:
    shutil.copytree(KAGGLE_DATASET_DIR, PROJECT_DIR)
else:
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)

os.chdir(PROJECT_DIR)
print("CWD:", os.getcwd())
print(os.listdir())

## 2. Cài dependencies

In [ ]:
!pip install -q -r requirements.txt

## 3. (Tuỳ chọn) HuggingFace token

Bỏ qua nếu không bị rate-limit. Để dùng: Add-ons → Secrets → thêm secret `HF_TOKEN`.

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded")
except Exception as e:
    print("Skip HF token:", e)

## 4. Kiểm tra GPU

In [ ]:
!nvidia-smi

## 5. Sinh dataset (~500 mẫu) và chia train/valid/test

In [ ]:
!python src/data/generate_dataset.py --target 500
!python src/data/generate_multi_turn.py --target 0 --output data/raw/dialogues_multi_turn.jsonl
!cat data/raw/dialogues.jsonl data/raw/dialogues_multi_turn.jsonl > data/raw/dialogues_merged.jsonl
!python src/data/split_data.py --input data/raw/dialogues_merged.jsonl
!ls -la data/processed/

## 6. Fine-tune QLoRA

Mặc định 3 epochs, batch 1 × grad-accum 8, LoRA r=8, NF4. Nếu OOM: sửa `configs/qwen_lora_sft.yaml` (`cutoff_len: 512`, `lora_rank: 4`, hoặc tăng `gradient_accumulation_steps`).

In [ ]:
!python src/train/train.py --config configs/qwen_lora_sft.yaml

## 7. So sánh base model vs fine-tuned trên test set

In [ ]:
!python -m src.inference.compare

In [ ]:
import json

with open("outputs/eval_results/comparison.jsonl", encoding="utf-8") as f:
    rows = [json.loads(l) for l in f]

print(f"Total: {len(rows)} samples\n")
for r in rows[:5]:
    print("DIALOGUE:", r["dialogue"])
    print("GOLD    :", r["gold"])
    print("BASE    :", r["base_output"])
    print("LORA    :", r["finetuned_output"])
    print("-" * 80)

## 8. Inference thử một hội thoại

In [ ]:
!python -m src.inference.predict \
  --conversation '[{"role":"user","content":"mở điều hoà"},{"role":"bot","content":"bạn muốn đặt bao nhiêu độ?"},{"role":"user","content":"27 độ"}]'

## 9. Lưu adapter ra `/kaggle/working` để download

Sau khi *Save Version* (Commit), mọi file trong `/kaggle/working` sẽ hiện trong tab Output của notebook.

In [ ]:
import shutil

src = "outputs/qwen-dialogue-rewriter-lora"
dst = "/kaggle/working/qwen-dialogue-rewriter-lora"
if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree(src, dst)

# Zip cho dễ download
shutil.make_archive("/kaggle/working/qwen-dialogue-rewriter-lora", "zip", src)
print("Saved adapter + zip to /kaggle/working/")
!ls -lh /kaggle/working/